<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/927_IRMv3_DataLoader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IRMv3 Data Loader

## Architecture Review & Explanation

The IRMv3 data loader is responsible for **collecting all operational telemetry required to evaluate the health of the AI ecosystem**.

Before the orchestrator can calculate scores or generate executive reports, it must first answer a basic question:

**What data describes the current state of the AI operation?**

This loader consolidates all of that information into a **single structured state object** that the rest of the orchestrator can analyze.

Instead of requiring multiple systems or APIs to be queried during execution, the loader reads a structured set of JSON files that represent the AI ecosystem’s telemetry.

This design makes the system:

* predictable
* transparent
* easy to test
* easy to audit

Those qualities are critical when the goal is **operational trust**.

---

# What the Data Loader Actually Does

At a practical level, the loader performs five key tasks:

1. **Locate the data directory**
2. **Load all telemetry files**
3. **Record dataset sizes**
4. **Build lookup structures for analysis**
5. **Validate the dataset and surface warnings**

Once complete, it returns a dictionary that contains everything needed for the rest of the orchestrator pipeline.

In effect, the loader produces the **initial system state**.

---

# Resilient File Loading

The `_load_json()` function provides a simple but important capability:

```python
def _load_json(path: Path) -> List[Dict[str, Any]]:
```

This function attempts to load a JSON file and returns an **empty list if anything goes wrong**.

The loader intentionally avoids throwing errors for:

* missing files
* corrupted JSON
* filesystem errors

Instead, the system continues operating with an empty dataset.

### Why this matters operationally

In real environments, telemetry systems are often imperfect.

Files may be delayed, partially generated, or temporarily unavailable.

A brittle loader would cause the orchestrator to fail entirely.

This implementation allows the orchestrator to **continue operating even when some telemetry sources are missing**.

That resilience is important for systems intended to monitor **mission-critical infrastructure**.

---

# Centralized Data Loading

The main function:

```python
load_all_irm_data()
```

acts as the **central ingestion layer** for the orchestrator.

It loads all telemetry datasets used by IRMv3:

### Structural system data

* agents
* workflows
* system_integrations
* dependency_graph

These describe **how the AI ecosystem is built**.

---

### Operational telemetry

* mission_runs
* task_execution_logs
* integration_health_logs
* dependency_incidents

These describe **how the system is behaving right now**.

---

### Governance signals

* risk_signals
* remediation_actions
* governance_reviews

These measure **how well the organization is responding to issues**.

---

### Financial performance

* expected_vs_actual_value
* kpis_cost

These allow the orchestrator to estimate **automation ROI**.

---

### Historical performance

* historical_snapshots

These enable **trend detection over time**.

---

# Why Centralized Loading Is Important

Many AI systems fetch data directly inside scoring logic.

That approach creates several problems:

* repeated file access
* unpredictable data flows
* difficult debugging
* inconsistent analysis

IRMv3 instead performs **all data loading upfront**.

The result is a **clean separation of responsibilities**:

| Component        | Responsibility            |
| ---------------- | ------------------------- |
| Data Loader      | Gather telemetry          |
| Scoring Engine   | Evaluate system health    |
| Rollup Logic     | Aggregate signals         |
| Report Generator | Produce executive insight |

This separation improves **maintainability and transparency**.

---

# Dataset Visibility Through Record Counts

The loader records how many records were loaded from each dataset.

```python
counts[key] = len(data)
```

This information becomes:

```python
loader_record_counts
```

This is a small but powerful design feature.

It allows the system to surface signals such as:

* incomplete datasets
* sudden drops in telemetry
* missing data sources

These are often early indicators of **observability failures**.

---

# Timestamped Data Snapshots

The loader records when the data snapshot was created.

```python
data_snapshot_loaded_at
```

This timestamp ensures every report can answer:

**What moment in time does this analysis represent?**

This is essential for:

* auditability
* incident investigation
* governance reviews

Without this information, operational intelligence systems become **difficult to trust**.

---

# Lookup Structures for Efficient Analysis

The loader also constructs several **derived lookup structures**.

These are designed to make the scoring stage more efficient and easier to reason about.

### Agent lookup

```python
agents_lookup
```

Maps:

```
agent_id → agent configuration
```

This allows the orchestrator to quickly retrieve metadata about a specific agent.

---

### Workflows by agent

```python
workflows_by_agent
```

Maps:

```
agent_id → workflows executed by that agent
```

This is useful for measuring:

* workflow performance
* automation value
* operational load by agent

---

### Workflows by integration

```python
workflows_by_integration
```

Maps:

```
integration_id → workflows dependent on that integration
```

This is particularly important for **dependency risk analysis**.

It allows the orchestrator to estimate **blast radius**.

Example insight:

> If this integration fails, how many workflows stop working?

That is exactly the type of operational risk signal leadership needs.

---

# Dependency Graph Processing

The loader converts the dependency graph into a more usable structure.

It identifies:

```
agent → integration dependencies
```

and then connects those integrations to:

```
workflows
```

This transformation allows the orchestrator to understand **how failures propagate across the system**.

Without this step, the dependency graph would remain a static dataset rather than a **usable operational model**.

---

# Data Validation Warnings

The loader performs a few basic validation checks.

For example:

```python
if counts.get("agents", 0) == 0
```

or

```python
if counts.get("integration_health_logs", 0) == 0
```

These checks produce warnings instead of errors.

This design choice reflects an important philosophy:

The orchestrator should **inform operators about data quality issues**, not fail silently or crash.

Warnings allow the system to surface **observability gaps** without interrupting operations.

---

# Why This Design Supports Executive Trust

For leaders responsible for AI operations, two concerns matter most:

1. **Is the system reliable?**
2. **Can we understand how conclusions are reached?**

This loader contributes to both.

It ensures that:

* all telemetry inputs are clearly defined
* missing data is explicitly surfaced
* every analysis is tied to a specific data snapshot
* the entire dataset can be inspected independently

That level of transparency makes the system **much easier to trust and govern**.

---

# How This Differs From Typical AI Agent Designs

Many AI systems today:

* fetch data dynamically
* rely on hidden pipelines
* mix loading logic with analysis
* produce results without preserving inputs

That approach can make systems fragile and difficult to audit.

IRMv3 instead treats telemetry ingestion as a **first-class architectural layer**.

By isolating the loader and building structured lookup maps, the orchestrator ensures that all later decisions are based on **clear, inspectable inputs**.

This approach aligns with the broader goal of the IRMv3 architecture:

**AI systems should be observable, governable, and explainable.**

---

# Potential Enhancements

Your loader is already well-designed, but a few upgrades could strengthen it further.

### Schema validation

Adding optional schema checks could ensure data consistency.

Example:

* missing required fields
* incorrect data types

---

### Duplicate detection

Detecting duplicate records in key datasets could improve reliability.

---

### Dependency integrity checks

You could verify that every dependency in `dependency_graph` references:

* a valid agent
* a valid integration

This would help detect **configuration drift**.

---

# Overall Assessment

This data loader is **clean, resilient, and architecturally sound**.

It demonstrates several strong engineering principles:

* clear separation of responsibilities
* resilient data ingestion
* structured telemetry modeling
* transparent data lineage
* operational observability

Rather than simply reading files, the loader establishes the **foundational dataset that allows the orchestrator to reason about the health of the entire AI ecosystem**.

That makes it one of the most important components in the IRMv3 system.


In [ ]:
"""Load all IRMv3 data from JSON files. Optional files return empty lists if missing."""

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional


def _load_json(path: Path) -> List[Dict[str, Any]]:
    """Load a JSON file; return [] if missing or invalid."""
    if not path.exists():
        return []
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, list) else []
    except (json.JSONDecodeError, OSError):
        return []


def load_all_irm_data(
    data_dir: str,
    project_root: Optional[str] = None,
    **file_names: str,
) -> Dict[str, Any]:
    """
    Load all IRM data files from data_dir (resolved against project_root if given).
    Returns dict with list keys for each dataset, loader_record_counts, data_snapshot_loaded_at,
    validation_warnings, and lookups (agents_lookup, workflows_by_agent, workflows_by_integration).
    Optional files: governance_reviews, expected_vs_actual_value, kpis_cost.
    """
    root = Path(project_root) if project_root else Path.cwd()
    base = root / data_dir
    base = base.resolve()

    out: Dict[str, Any] = {}
    counts: Dict[str, int] = {}

    # Required / expected files
    files = {
        "agents": file_names.get("agents_file", "agents.json"),
        "workflows": file_names.get("workflows_file", "workflows.json"),
        "system_integrations": file_names.get("system_integrations_file", "system_integrations.json"),
        "dependency_graph": file_names.get("dependency_graph_file", "dependency_graph.json"),
        "mission_runs": file_names.get("mission_runs_file", "mission_runs.json"),
        "task_execution_logs": file_names.get("task_execution_logs_file", "task_execution_logs.json"),
        "integration_health_logs": file_names.get("integration_health_logs_file", "integration_health_logs.json"),
        "dependency_incidents": file_names.get("dependency_incidents_file", "dependency_incidents.json"),
        "risk_signals": file_names.get("risk_signals_file", "risk_signals.json"),
        "remediation_actions": file_names.get("remediation_actions_file", "remediation_actions.json"),
        "historical_snapshots": file_names.get("historical_snapshots_file", "historical_snapshots.json"),
        "governance_reviews": file_names.get("governance_reviews_file", "governance_reviews.json"),
        "expected_vs_actual_value": file_names.get("expected_vs_actual_value_file", "expected_vs_actual_value.json"),
        "kpis_cost": file_names.get("kpis_cost_file", "kpis_cost.json"),
    }

    for key, filename in files.items():
        path = base / filename
        data = _load_json(path)
        out[key] = data
        counts[key] = len(data)

    out["loader_record_counts"] = counts
    out["data_snapshot_loaded_at"] = datetime.now(timezone.utc).isoformat()

    # Build lookups
    agents = out["agents"]
    workflows = out["workflows"]
    dependency_graph = out["dependency_graph"]

    out["agents_lookup"] = {a["agent_id"]: a for a in agents} if agents else {}

    workflows_by_agent: Dict[str, List[Dict[str, Any]]] = {}
    for w in workflows:
        aid = w.get("agent_id")
        if aid:
            workflows_by_agent.setdefault(aid, []).append(w)
    out["workflows_by_agent"] = workflows_by_agent

    # integration_id -> list of workflow_ids that depend on it (via agent -> dependency)
    # From dependency_graph: source_component = agent_id, target_component = integration_id.
    integration_to_workflows: Dict[str, List[str]] = {}
    agent_to_integrations: Dict[str, List[str]] = {}
    for edge in dependency_graph:
        src = edge.get("source_component")
        tgt = edge.get("target_component")
        if not src or not tgt:
            continue
        agent_to_integrations.setdefault(src, []).append(tgt)
    for agent_id, int_ids in agent_to_integrations.items():
        wf_ids = [w["workflow_id"] for w in workflows_by_agent.get(agent_id, [])]
        for iid in int_ids:
            for wid in wf_ids:
                integration_to_workflows.setdefault(iid, []).append(wid)
    # Deduplicate
    for iid in integration_to_workflows:
        integration_to_workflows[iid] = list(dict.fromkeys(integration_to_workflows[iid]))
    out["workflows_by_integration"] = integration_to_workflows

    # Validation warnings
    warnings: List[str] = []
    if counts.get("agents", 0) == 0:
        warnings.append("No agents loaded.")
    if counts.get("integration_health_logs", 0) == 0:
        warnings.append("No integration_health_logs; integration stability score will use defaults.")
    out["validation_warnings"] = warnings

    return out
